This notebook is for modeling and evaluating technique classification from the SemEval dataset. Given a span predicted to be propaganda, this model will identify which propaganda techniques, if any, were used.

Initially, we developed a **Hybrid RoBERTa Model** to incorporate manual linguistic features alongside deep learning embeddings. The goal was to determine if explicit metadata could assist the model in identifying complex propaganda techniques.

**Linguistic Features Added:**
1. **Sentiment Analysis:** Polarity and subjectivity scores (via TextBlob).
2. **Punctuation Density:** Count of exclamation marks, question marks, and CAPS, which are often markers of emotional "Loaded Language."
3. **Lexical Diversity:** Type-Token Ratio (TTR) to measure the complexity and variety of the vocabulary used in the span.

**Ablation Study Results:**
We performed an ablation study by comparing the Hybrid model's performance against a simple text baseline (where manual features were zeroed out).
* **Hybrid Micro-F1:** 0.1001
* **Text-Only Micro-F1:** 0.0987
* **Total Feature Lift:** +0.0014 (< 0.2%)

**Conclusion:**
The manual features provided a negligible lift of only **0.14%**. This indicates that the RoBERTa backbone is already capturing these linguistic nuances internally through its attention mechanisms.

**Decision:** To prioritize **model replicability** and **pipeline stability**, we have transitioned to a standard `AutoModelForSequenceClassification`. This allows for:
* **Native Hugging Face Support:** Automatic generation of `config.json` and `id2label` mappings.
* **Portability:** The model can be loaded in one line without requiring custom Python class definitions in downstream notebooks.

We tuned hyperparameters to prioritize maximizing overall macro F1 score, since we want to be careful to learn all the labels and not just the majority ones. Then, using those hyperparameters, did a final training optimizing micro F1 score.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import GroupShuffleSplit
from transformers.utils.notebook import NotebookProgressCallback
from safetensors.torch import load_file
import gdown
import zipfile
import shutil

In [2]:
#Define global paths
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"
TC_MODEL_PATH = MODEL_DIR / "semeval_roberta_classifier"

In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
special_tokens_dict = {'additional_special_tokens': ['[SPAN]', '[/SPAN]']}
tokenizer.add_special_tokens(special_tokens_dict)
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)

In [4]:
google_drive_tc_zip_ID = '1YiyjCg0quWmeJkMm5RcM0xKjOanIwz_w'
google_drive_tc_context_zip_ID = '1PZXDKzsYcWGRE8U6nGSqq7dDi3F1TpSh'

In [5]:
#Download models if not already
def setup_models(file_id, target_path):
    target_path = Path(target_path).resolve()
    zip_temp = target_path.with_suffix(".zip")

    #Check if files already exist in the correct spot
    if (target_path / "model.safetensors").exists() or (target_path / "pytorch_model.bin").exists():
        print(f"Model weights detected locally at {target_path}")
        return True

    print(f"Model not found. Preparing {target_path}...")

    #Ensure the specific sub-folder exists
    target_path.mkdir(exist_ok=True, parents=True)

    url = f'https://drive.google.com/uc?id={file_id}'

    try:
        #1. Download the zip
        gdown.download(url, str(zip_temp), quiet=False)

        #2. Extract to a temporary location
        temp_extract = target_path / "temp_extraction"
        if temp_extract.exists(): shutil.rmtree(temp_extract)
        temp_extract.mkdir(parents=True)

        print("Unzipping and cleaning up structure...")
        with zipfile.ZipFile(zip_temp, 'r') as zip_ref:
            members = [m for m in zip_ref.namelist() if "__MACOSX" not in m]
            zip_ref.extractall(temp_extract, members=members)

        #3. Move files from temp_extract into target_path
        for root, dirs, files in os.walk(temp_extract):
            for file in files:
                src_file = Path(root) / file
                dest_file = target_path / file
                shutil.move(str(src_file), str(dest_file))

        #4. Final Cleanup
        shutil.rmtree(temp_extract)
        if zip_temp.exists():
            os.remove(zip_temp)

        print(f"Model files are now in: {target_path}")
        return True

    except Exception as e:
        print(f"Error during setup: {e}")
        if zip_temp.exists(): os.remove(zip_temp)
        return False

#Identify if model exists
model_already_trained = setup_models(google_drive_tc_context_zip_ID, TC_MODEL_PATH)

Model not found. Preparing /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_classifier...


Downloading...
From (original): https://drive.google.com/uc?id=1PZXDKzsYcWGRE8U6nGSqq7dDi3F1TpSh
From (redirected): https://drive.google.com/uc?id=1PZXDKzsYcWGRE8U6nGSqq7dDi3F1TpSh&confirm=t&uuid=8b631d0d-4f8f-4a9f-aaf4-6316c4792b2f
To: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_classifier.zip
100%|██████████| 444M/444M [00:10<00:00, 40.5MB/s] 


Unzipping and cleaning up structure...
Model files are now in: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_classifier


In [6]:
#Load technique classification data
df_tc = pd.read_csv(DATA_DIR / "semeval_tc_cleaned.csv")
df_tc.head()

,article_id,text_content,span_text,start_char,end_char,sentiment,punct_count,lexical_diversity,Appeal_to_Authority,Appeal_to_fear-prejudice,...,Loaded_Language,Minimisation,Name_Calling,Red_Herring,Reductio_ad_hitlerum,Repetition,Slogans,Straw_Men,Thought-terminating_Cliches,Whataboutism
0,111111111,Next plague outbreak in Madagascar could be 's...,appeared,149,157,0.000000,0,1.000000,0,0,...,0,0,0,0,0,0,0,0,0,0
1,111111111,Next plague outbreak in Madagascar could be 's...,The next transmission could be more pronounced...,265,323,0.250000,0,1.000000,1,0,...,0,0,0,0,0,0,0,0,0,0
2,111111111,Next plague outbreak in Madagascar could be 's...,"a very, very different",1069,1091,0.000000,0,1.000000,0,0,...,0,0,0,0,0,1,0,0,0,0
3,111111111,Next plague outbreak in Madagascar could be 's...,He also pointed to the presence of the pneumon...,1334,1462,0.483333,0,0.863636,0,1,...,0,0,0,0,0,0,0,0,0,0
4,111111111,Next plague outbreak in Madagascar could be 's...,but warned that the danger was not over,1577,1616,0.000000,0,1.000000,0,1,...,0,0,0,0,0,0,0,0,0,0


In [7]:
#Split the data by article rather than by span to avoid any data leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_tc, groups=df_tc['article_id']))
train_df = df_tc.iloc[train_idx].reset_index(drop=True)
test_df = df_tc.iloc[test_idx].reset_index(drop=True)

#Verify the split
print(f"Total spans: {len(df_tc)}, total articles: {df_tc['article_id'].nunique()}")
print(f"Train spans: {len(train_df)} ({train_df['article_id'].nunique()} articles)")
print(f"Test spans:  {len(test_df)} ({test_df['article_id'].nunique()} articles)")

#Check for leakage (should be 0)
overlap = set(train_df['article_id']).intersection(set(test_df['article_id']))
print(f"Number of overlapping articles: {len(overlap)}")

Total spans: 7587, total articles: 357
Train spans: 5856 (285 articles)
Test spans:  1731 (72 articles)
Number of overlapping articles: 0


In [8]:
#Identify feature columns versus input versus output
feature_cols = ['sentiment', 'punct_count', 'lexical_diversity']
label_cols = [c for c in train_df.columns if c not in (['article_id', 'text_content', 'span_text', 'start_char', 'end_char'] + feature_cols)]
print(label_cols)

['Appeal_to_Authority', 'Appeal_to_fear-prejudice', 'Bandwagon', 'Black-and-White_Fallacy', 'Causal_Oversimplification', 'Doubt', 'Exaggeration', 'Flag-Waving', 'Labeling', 'Loaded_Language', 'Minimisation', 'Name_Calling', 'Red_Herring', 'Reductio_ad_hitlerum', 'Repetition', 'Slogans', 'Straw_Men', 'Thought-terminating_Cliches', 'Whataboutism']


In [9]:
#Ensure all the label columns exist in the dataframe
missing = [c for c in label_cols if c not in train_df.columns]
if missing:
    print(f"Warning: Missing columns in dataframe: {missing}")
else:
    print(f"Label columns correctly set. Number of techniques: {len(label_cols)}")

Label columns correctly set. Number of techniques: 19


In [10]:
#Cast labels to floats because BCEWithLogitsLoss requires float tensors, not integers
train_df['labels'] = train_df[label_cols].astype(float).values.tolist()
test_df['labels'] = test_df[label_cols].astype(float).values.tolist()

Using all 19 techniques in the dataset instead of the 14 they were constrained to in the competition to get more granular results that may be more useful and specific for the user. For this reason, scores may be slightly lower than they would otherwise be because it's a more difficult task. In the competition, Bandwagon and Reductio ad hitlerum were combined into one category, Exaggeration and Minimisation were flattened to one, Name_Calling and Labeling were flattened to one, and Whataboutism, Straw_Men, and Red_Herring were collapsed to one.

In [12]:
#Put input text data in a format Hugging Face knows how to use
WINDOW_SIZE = 200

def preprocess_function(examples):
    contextualized_inputs = []

    for i in range(len(examples["span_text"])):
        span = str(examples["span_text"][i])
        full_text = str(examples["text_content"][i])

        #Get the exact character positions
        start_idx = int(examples["start_char"][i])
        end_idx = int(examples["end_char"][i])

        #Slice the string to get the text immediately before and after the span
        before_context = full_text[max(0, start_idx - WINDOW_SIZE) : start_idx]

        #min(len(), ...) prevents errors if the span is at the very end
        after_context = full_text[end_idx : min(len(full_text), end_idx + WINDOW_SIZE)]

        #Reconstruct the string with our markers
        marked_text = f"{before_context} [SPAN] {span} [/SPAN] {after_context}"
        contextualized_inputs.append(marked_text)

    #Tokenize the newly windowed strings
    return tokenizer(
        contextualized_inputs,
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_train = Dataset.from_pandas(train_df).map(preprocess_function, batched=True)
tokenized_test = Dataset.from_pandas(test_df).map(preprocess_function, batched=True)

Map:   0%|          | 0/5856 [00:00<?, ? examples/s]

Map:   0%|          | 0/1731 [00:00<?, ? examples/s]

In [13]:
#Create the config so the model knows the names of the techniques
config = AutoConfig.from_pretrained(
    "roberta-base",
    num_labels=len(label_cols),
    id2label={i: label for i, label in enumerate(label_cols)},
    label2id={label: i for i, label in enumerate(label_cols)},
    problem_type="multi_label_classification",
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2
)

In [14]:
#Load the standard model
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    config=config
).to(device)
model.resize_token_embeddings(len(tokenizer))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and

Embedding(50267, 768, padding_idx=1)

In [15]:
#If model already exists, load those weights
if model_already_trained:
    safe_path = TC_MODEL_PATH / "model.safetensors"
    bin_path = TC_MODEL_PATH / "pytorch_model.bin"

    state_dict = None

    if safe_path.exists():
        state_dict = load_file(safe_path, device="cpu")
    elif bin_path.exists():
        state_dict = torch.load(bin_path, map_location=torch.device('cpu'))

    if state_dict:
        new_state_dict = {}
        for key, value in state_dict.items():
            new_key = key.replace(".beta", ".bias").replace(".gamma", ".weight")
            new_state_dict[new_key] = value

        model.load_state_dict(new_state_dict, strict=True)
        print("Weights successfully renamed and injected into the model.")
    else:
        print(f"Error: No weights found. Check your paths!")

Weights successfully renamed and injected into the model.


In [16]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        #1. POP the labels so they don't get passed to the model's internal loss function
        labels = inputs.pop("labels")

        #2. Get model predictions (logits)
        outputs = model(**inputs)
        logits = outputs.get("logits")

        #3. Initialize standard BCE loss (No weighting, no smoothing)
        loss_fct = nn.BCEWithLogitsLoss()

        #4. Calculate the final loss using the exact, raw labels
        #must cast labels to .float() because BCEWithLogits expects floats, not ints
        loss = loss_fct(logits, labels.float())

        return (loss, outputs) if return_outputs else loss

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    predictions = (probs > 0.5).astype(int)
    labels = labels.astype(int)

    return {
        "precision": precision_score(labels, predictions, average="micro", zero_division=0),
        "recall": recall_score(labels, predictions, average="micro", zero_division=0),
        "f1": f1_score(labels, predictions, average="micro", zero_division=0),
        "samples_f1": f1_score(labels, predictions, average="samples", zero_division=0),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
    }

In [18]:
training_args = TrainingArguments(
    output_dir=TC_MODEL_PATH,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2.47e-05,
    per_device_train_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.2,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    load_best_model_at_end=True
)

In [19]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

In [20]:
#Train only if we didn't load a local model
if not model_already_trained:
    print("Starting training process...")
    trainer.train()
    trainer.save_model(TC_MODEL_PATH)
    tokenizer.save_pretrained(os.fspath(TC_MODEL_PATH))
    print(f"Model trained and saved to {MODEL_DIR}")
else:
    print("Model loaded from disk. Skipping training.")

Model loaded from disk. Skipping training.


In [21]:
#Evaluate performance on the test dataset
trainer.remove_callback(NotebookProgressCallback)
test_results = trainer.evaluate(eval_dataset=tokenized_test)

print("\n" + "="*30)
print("FINAL MODEL PERFORMANCE")
print(f"Recall:    {test_results['eval_recall']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"F1 Score:  {test_results['eval_f1']:.4f}")
print(f"Macro F1 Score:  {test_results['eval_macro_f1']:.4f}")
print("="*30)

/Users/frankiepike/ds_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



FINAL MODEL PERFORMANCE
Recall:    0.5907
Precision: 0.7332
F1 Score:  0.6543
Macro F1 Score:  0.3387


Previous best performance, simple RoBERTa setup:
FINAL MODEL PERFORMANCE
Recall:    0.7452
Precision: 0.4818
F1 Score:  0.5852
Macro F1 Score:  0.4544

Weighting by inverse frequency tanked performance by just guessing all the rare techniques constantly. Smoothing had the same impact.

Best performance with considering additional tokens for more context around each span (context injection): FINAL MODEL PERFORMANCE
Recall:    0.5907
Precision: 0.7332
F1 Score:  0.6543
Macro F1 Score:  0.3387

In [22]:
#Have the trainer predict on the test set
#This returns an object containing 'predictions' (logits) and 'label_ids' (true labels)
test_predictions = trainer.predict(tokenized_test)

#Extract the raw logits and true labels
logits = torch.tensor(test_predictions.predictions)
true_labels = test_predictions.label_ids

#Apply Sigmoid to convert logits (raw math) into probabilities (0.0 to 1.0)
probabilities = torch.sigmoid(logits).numpy()

#Apply a threshold to turn probabilities into binary Yes/No predictions
THRESHOLD = 0.5
binary_predictions = (probabilities >= THRESHOLD).astype(int)

#Generate and print the Table 6 equivalent!
print("--- Per-Technique Performance (Table 6 Equivalent) ---")
print(classification_report(
    true_labels,
    binary_predictions,
    target_names=label_cols,
    zero_division=0
))

/Users/frankiepike/ds_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


--- Per-Technique Performance (Table 6 Equivalent) ---
                             precision    recall  f1-score   support

        Appeal_to_Authority       0.00      0.00      0.00        18
   Appeal_to_fear-prejudice       0.52      0.52      0.52        61
                  Bandwagon       0.00      0.00      0.00        18
    Black-and-White_Fallacy       0.00      0.00      0.00        33
  Causal_Oversimplification       0.50      0.04      0.08        49
                      Doubt       0.67      0.57      0.62       102
               Exaggeration       0.52      0.45      0.48       101
                Flag-Waving       0.72      0.75      0.73        64
                   Labeling       0.81      0.77      0.79       219
            Loaded_Language       0.82      0.80      0.81       455
               Minimisation       0.52      0.45      0.48       101
               Name_Calling       0.81      0.77      0.79       219
                Red_Herring       1.00      0.0